# Jurimetría Asistida por IA
## Análisis de sentencias contencioso-administrativas — Tribunal de Guayas

Este cuaderno implementa el modelo pedagógico de tres fases:
1. **Recopilar** sentencias desde EXPEL
2. **Extraer** variables con IA (Claude)
3. **Analizar** y exportar a CSV

In [ ]:
# PASO 0 — Instalar dependencias
!pip install -q requests anthropic

In [ ]:
# PASO 1 — Tu clave de API de Anthropic
# Ve a console.anthropic.com → API Keys → copia tu clave
import os
ANTHROPIC_API_KEY = "sk-ant-..."  # <-- pega tu clave aquí
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

In [ ]:
# PASO 2 — Descargar sentencias desde EXPEL (Función Judicial)
import requests
import json

BASE_URL = "https://procesosjudiciales.funcionjudicial.gob.ec"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": BASE_URL,
    "Referer": BASE_URL + "/transaccion-consulta-numero-causa"
}

def buscar_causas_expel(provincia="GUAYAS", materia="CONTENCIOSO ADMINISTRATIVO", max_resultados=20):
    url = BASE_URL + "/transaccion-consulta-numero-causa"
    payload = {
        "numeroCausa": "",
        "actor": {"nombre": ""},
        "demandado": {"nombre": ""},
        "provincia": provincia,
        "materia": materia,
        "recaptcha": "token"
    }
    try:
        resp = requests.post(url, json=payload, headers=HEADERS, timeout=30)
        print(f"Status: {resp.status_code}")
        if resp.status_code == 200:
            return resp.json()
        else:
            print(f"Error: {resp.text[:300]}")
            return []
    except Exception as e:
        print(f"Error de conexión: {e}")
        return []

resultados = buscar_causas_expel()
print(f"Causas encontradas: {len(resultados)}")

In [ ]:
# PASO 2B — Si EXPEL no responde, usa estos datos de ejemplo para continuar
# (datos reales de estructura típica del Tribunal Contencioso de Guayas)

sentencias_ejemplo = [
  {"numero": "17811-2019-00234", "tipo": "Contencioso Administrativo", "fecha": "15/03/2021",
   "ponente": "Dr. Carlos Andrade Mora",
   "resumen": "Proceso contencioso administrativo de Maria Rodriguez contra el Municipio de Guayaquil. Impugna destitucion de cargo de inspectora municipal. Demanda presentada el 10 de enero de 2019. Etapa de prueba del 5 de marzo al 4 de junio de 2019, duracion 90 dias. Tribunal Distrital de Guayas resolvio el 15 de marzo de 2021: con lugar, se ordena restitucion al cargo por vicios de procedimiento. Derechos invocados: debido proceso, derecho al trabajo."},
  {"numero": "17811-2020-00089", "tipo": "Contencioso Administrativo", "fecha": "22/08/2022",
   "ponente": "Dra. Patricia Villacis Torres",
   "resumen": "Accion contencioso administrativa de CONSTRUVIAL S.A. contra el Gobierno Provincial del Guayas. Impugna terminacion unilateral de contrato de obra publica. Demanda del 3 de febrero de 2020. Etapa de prueba del 12 de mayo al 10 de agosto de 2020, duracion 90 dias. Sentencia del 22 de agosto de 2022: sin lugar, no se acreditaron perjuicios. Derechos invocados: seguridad juridica, debido proceso."},
  {"numero": "17811-2018-00412", "tipo": "Contencioso Administrativo", "fecha": "07/11/2022",
   "ponente": "Dr. Roberto Espinoza Lara",
   "resumen": "Demanda de Juan Perez Neira contra la Prefectura del Guayas. Impugna negativa de reconocimiento de tiempo de servicios para jubilacion. Presentada el 18 de julio de 2018. Etapa de prueba del 3 de octubre al 31 de diciembre de 2018, duracion 89 dias. Resolucion del 7 de noviembre de 2022: con lugar, se ordena reliquidacion de pension jubilar. Derechos invocados: seguridad social, debido proceso."},
  {"numero": "17811-2021-00156", "tipo": "Contencioso Administrativo", "fecha": "14/04/2023",
   "ponente": "Dra. Patricia Villacis Torres",
   "resumen": "Andrea Suarez contra la Contraloria General del Estado zona Guayas. Impugna glosa por perjuicio al Estado en obra de alcantarillado. Demanda del 22 de marzo de 2021. Etapa de prueba del 10 de junio al 7 de septiembre de 2021, duracion 89 dias. Sentencia del 14 de abril de 2023: parcialmente con lugar, se deja sin efecto la glosa personal. Derechos invocados: debido proceso, derecho a la defensa."},
  {"numero": "17811-2019-00578", "tipo": "Contencioso Administrativo", "fecha": "30/06/2023",
   "ponente": "Dr. Carlos Andrade Mora",
   "resumen": "Luis Mendez Gallardo contra el Ministerio de Salud Publica Coordinacion Zonal 8. Impugna destitucion de medico rural. Presentada el 15 de agosto de 2019. Etapa probatoria del 2 de noviembre de 2019 al 30 de enero de 2020, duracion 89 dias. Resolucion del 30 de junio de 2023: sin lugar, proceso disciplinario cumplio garantias. Derechos invocados: debido proceso, estabilidad laboral."},
  {"numero": "17811-2020-00334", "tipo": "Contencioso Administrativo", "fecha": "19/09/2022",
   "ponente": "Dr. Roberto Espinoza Lara",
   "resumen": "IMPORTEC S.A. contra la Aduana del Ecuador Distrito Guayaquil. Impugna tributos adicionales por importacion de maquinaria. Demanda del 11 de mayo de 2020. Etapa de prueba del 3 de agosto al 31 de octubre de 2020, duracion 89 dias. Sentencia del 19 de septiembre de 2022: sin lugar. Derechos invocados: seguridad juridica, libre empresa."},
  {"numero": "17811-2017-00891", "tipo": "Contencioso Administrativo", "fecha": "28/02/2023",
   "ponente": "Dra. Carmen Holguin Vera",
   "resumen": "Rosa Villafuerte contra el Consejo de la Judicatura Guayas. Impugna no renovacion de contrato ocasional. Presentada el 4 de marzo de 2017. Etapa de prueba del 18 de mayo al 15 de agosto de 2017, duracion 89 dias. Resolucion del 28 de febrero de 2023: con lugar por vulneracion de confianza legitima. Derechos invocados: estabilidad laboral, igualdad."},
  {"numero": "17811-2022-00045", "tipo": "Contencioso Administrativo", "fecha": "11/01/2024",
   "ponente": "Dra. Carmen Holguin Vera",
   "resumen": "Pedro Aguirre Mora contra el SRI Regional Litoral. Impugna acto determinativo de obligacion tributaria. Presentada el 28 de enero de 2022. Etapa probatoria del 15 de abril al 13 de julio de 2022, duracion 89 dias. Sentencia del 11 de enero de 2024: con lugar, acto sin efecto por vicios en notificacion. Derechos invocados: debido proceso, seguridad juridica."},
  {"numero": "17811-2018-00667", "tipo": "Contencioso Administrativo", "fecha": "05/07/2021",
   "ponente": "Dr. Carlos Andrade Mora",
   "resumen": "Transportes Ecuatorianos contra la Agencia Nacional de Transito Guayas. Impugna negativa de renovacion de cupos. Presentada el 20 de septiembre de 2018. Etapa de prueba del 5 de diciembre de 2018 al 4 de marzo de 2019, duracion 90 dias. Resolucion del 5 de julio de 2021: sin lugar. Derechos invocados: libre empresa, seguridad juridica."},
  {"numero": "17811-2021-00423", "tipo": "Contencioso Administrativo", "fecha": "03/03/2024",
   "ponente": "Dra. Patricia Villacis Torres",
   "resumen": "Gabriela Torres Rios contra la Secretaria del Agua Guayas. Impugna revocacion de concesion de agua agricola. Presentada el 7 de julio de 2021. Etapa probatoria del 20 de septiembre al 18 de diciembre de 2021, duracion 89 dias. Sentencia del 3 de marzo de 2024: con lugar por vicio de motivacion. Derechos invocados: derecho al agua, seguridad juridica."}
]

# Usar datos de EXPEL si se obtuvieron, si no usar ejemplos
sentencias = resultados if resultados else sentencias_ejemplo
print(f"Trabajando con {len(sentencias)} sentencias")

In [ ]:
# PASO 3 — Extraer variables con IA (Claude)
import anthropic
import re
import time

client = anthropic.Anthropic()

PROMPT = """
Eres un asistente juridico. Analiza esta sentencia y extrae las variables en JSON.

- numero_causa: identificador del proceso
- tipo_accion: tipo de accion o recurso
- organo_jurisdiccional: tribunal que resuelve
- fecha_inicio: fecha de presentacion (YYYY-MM-DD o null)
- fecha_resolucion: fecha de sentencia (YYYY-MM-DD o null)
- duracion_dias: dias entre inicio y resolucion (entero o null)
- decision: favorable | desfavorable | parcial | inadmitida | no determinado
- derecho_invocado: lista de derechos invocados
- materia: area del derecho
- etapa_prueba_dias: dias de etapa probatoria (entero o null)

Responde SOLO con JSON.

TEXTO: {texto}
"""

def extraer(texto):
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        messages=[{"role": "user", "content": PROMPT.format(texto=texto[:8000])}]
    )
    raw = msg.content[0].text
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        return json.loads(match.group())
    return {}

enriquecidas = []
for i, s in enumerate(sentencias, 1):
    texto = s.get("resumen", "")
    print(f"[{i}/{len(sentencias)}] Procesando {s.get('numero', '')}...")
    variables = extraer(texto)
    enriquecidas.append({**s, **variables})
    time.sleep(0.5)

print("\nExtraccion completada.")

In [ ]:
# PASO 4 — Análisis descriptivo
import statistics
from collections import Counter

n = len(enriquecidas)
print(f"Total sentencias analizadas: {n}")
print()

# Decisiones
decisiones = Counter(s.get("decision", "no determinado") for s in enriquecidas)
print("DECISION:")
for d, c in decisiones.most_common():
    print(f"  {d}: {c} ({c/n*100:.0f}%)")
print()

# Por juez/ponente
jueces = Counter(s.get("ponente", "desconocido") for s in enriquecidas)
print("POR JUEZ:")
for j, c in jueces.most_common():
    print(f"  {j}: {c} causas")
print()

# Duracion etapa de prueba por juez
por_juez = {}
for s in enriquecidas:
    dias = s.get("etapa_prueba_dias")
    juez = s.get("ponente", "desconocido")
    if dias and str(dias).isdigit():
        por_juez.setdefault(juez, []).append(int(dias))

if por_juez:
    print("DURACION ETAPA DE PRUEBA POR JUEZ (dias):")
    for juez, vals in sorted(por_juez.items(), key=lambda x: -statistics.mean(x[1])):
        print(f"  {juez}: promedio {statistics.mean(vals):.0f} dias (n={len(vals)})")
print()

# Derechos mas invocados
derechos = Counter()
for s in enriquecidas:
    for d in (s.get("derecho_invocado") or []):
        derechos[d] += 1
print("DERECHOS MAS INVOCADOS:")
for d, c in derechos.most_common(5):
    print(f"  {d}: {c}")

In [ ]:
# PASO 5 — Exportar CSV
import csv
from google.colab import files

columnas = ["numero", "tipo", "fecha", "ponente", "tipo_accion", "organo_jurisdiccional",
            "fecha_inicio", "fecha_resolucion", "duracion_dias", "decision",
            "derecho_invocado", "materia", "etapa_prueba_dias"]

with open("jurimetria_guayas.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=columnas, extrasaction="ignore")
    w.writeheader()
    for s in enriquecidas:
        fila = {}
        for col in columnas:
            v = s.get(col, "")
            fila[col] = "; ".join(v) if isinstance(v, list) else (str(v) if v else "")
        w.writerow(fila)

print("CSV generado: jurimetria_guayas.csv")
files.download("jurimetria_guayas.csv")

## Preguntas para la discusion juridica (Fase 3)

1. Que explica el patron de decisiones a favor vs en contra?
2. Hay variacion en la duracion de la etapa de prueba segun el juez? Que explicacion juridica tiene?
3. Que variables NO captura este modelo?
4. Que riesgos eticos tiene extrapolar estos promedios a un caso individual?
5. Que casos quedaron fuera de esta muestra y por que?